# MoT Stage 2 — GPU ablation

Runs the real stage-2 comparison (spec §8): ~50-100M param models, full-size streamed datasets, on a free GPU tier.

**Before running:** Runtime → Change runtime type → GPU (T4 is fine).

This trains one arm per run through the notebook (MoT, baseline, or SOTA/cl100k_base) - set `ARM` below. Each arm needs its own GPU time budget; Colab's free tier gives you a handful of hours/day, so plan on separate sessions per arm unless you have Colab Pro.

In [ ]:
ARM = "mot"  # "mot" | "baseline" | "sota"

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU - set Runtime > Change runtime type > GPU before continuing'

## 2. Get the repo

Replace the URL if you've pushed local changes since this notebook was generated.

In [ ]:
!git clone https://github.com/karthik-sys/generalization_through_tokenization.git repo
%cd repo
!pip install -q -r requirements.txt

## 3. Hugging Face login (needed for the gated `the-stack-v2-dedup` code source)

Accept the dataset's terms at https://huggingface.co/datasets/bigcode/the-stack-v2-dedup first, then run this cell and paste a token from https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import login
login()

## 4. Mount Drive for checkpoint persistence

Colab sessions are ephemeral - without this, checkpoints vanish when the runtime recycles.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/mot_checkpoints', exist_ok=True)
!ln -sfn /content/drive/MyDrive/mot_checkpoints checkpoints

## 5. Sample data for tokenizer training

Pulls ~50k streamed docs per domain (not the full corpus - training itself streams live) to train stage-2-sized tokenizers. Only needs to run once; re-run if you want fresh tokenizers.

In [ ]:
!python3 src/data/stage2_sample_for_tokenizers.py

## 6. Train stage-2 tokenizers (24k-48k vocab, see src/model/stage2_config.py)

In [ ]:
!python3 src/tokenizers/train_all_stage2.py

## 7. Train

Streams live from HF the whole time - no full dataset ever hits disk. Checkpoints save to Drive every `CHECKPOINT_EVERY` steps (src/model/stage2_config.py) so you can resume after a Colab disconnect.

In [ ]:
!python3 src/train_stage2.py {ARM} --tokenizer-dir tokenizers_stage2

## Notes

- `MAX_STEPS`, `BATCH_SIZE`, `CHECKPOINT_EVERY` etc. are in `src/model/stage2_config.py` - edit before running if you want a different time budget or your GPU has more/less VRAM than a T4.
- MoT lands at ~89.1M params, the unified BPE baseline at ~68.6M, cl100k_base (SOTA) at whatever `BaselineModel(vocab_size=100277, ...)` comes out to - the script prints exact counts at startup.
- Domain tags (`<domain:X>`) are literal text visible to every arm (fairness rule, docs/dataset_methodology.md) - only MoT has a mechanism to act on them.
- If a session disconnects mid-run, reload the latest `checkpoints/*_step*.pt` from Drive and resume (resume logic isn't wired into train_stage2.py yet - add an `--resume <path>` load before the training loop if you need it).